In [2]:
# import  os
# num_cores = "1"
# os.environ["OPENBLAS_NUM_THREADS"] = num_cores
# os.environ["OMP_NUM_THREADS"] = num_cores
# os.environ["MKL_NUM_THREADS"] = num_cores

In [2]:
import sys
sys.path.append('../')

import numpy as np
import scipy.sparse as ssp
import matplotlib.pyplot as plt
import qutip as qt
import scqubits as scq
from matplotlib.colors import LogNorm
from tqdm import tqdm
from qutip.qip.operations import rz, cz_gate
import cmath
from sympy import symbols
from scipy.sparse.linalg import eigsh
import utils_2Q_gate_zp as ut
from joblib import Parallel, delayed
from IPython.display import display, Math
# ut.set_fig_font() ### Set various sizes in plotting
import networkx as nx
from multiprocessing import Pool
import pandas as pd

In [7]:
n_full = 500
n_truc = 500 # number of states in the graph model
gate = 'cnot' # 'x_gate_theta', 'x_gate_phi', 'cz', ''cnot
[hspace_full, _, eval_tot, n_theta0_dress, n_theta1_dress, 
    _, _, logi_state] = ut.load_qubit_data_2q(n_full)
if gate == 'cnot':
    A = [0.02, 0.02]
    drive_term = n_theta0_dress
    state_mid = '8-2'
    idx_0 = hspace_full.index('0-2')
    idx_1 = hspace_full.index('2-2')
    idx_2 = hspace_full.index(state_mid)
    core_states = logi_state + [state_mid]
    W_0_2 = eval_tot[idx_2] - eval_tot[idx_0]
    W_1_2 = eval_tot[idx_2] - eval_tot[idx_1]
    wd = [W_0_2, W_1_2]
elif gate == 'cz':
    A = [0.015]
    detune = 0 # 0.023
    drive_term = n_theta1_dress
    state_mid = '5-0'
    core_states = logi_state + [state_mid]
    wd = (eval_tot[hspace_full.index(state_mid)] 
            - eval_tot[hspace_full.index('2-0')] 
            + 2 * np.pi * detune)

In [ ]:
n = n_truc
labels=hspace_full
path_func=ut.shortest_path_to_core

if isinstance(wd, float):
    wd = [wd]
if isinstance(A, float):
    A = [A]

df_list = []
for i in range(len(wd)):
    df = ut.make_leakage_df(core_states, drive_term, eval_tot, wd[i], A[i], 
                            G=None, labels=labels, path_func=path_func)
    df_list.append(df)
# print(f'np.shape(df_list)={np.shape(df_list)}')
# print(f'df_list={df_list}')
df = pd.concat(
                df_list
                ).sort_values(
                                "path_len", ascending=False
                                ).drop_duplicates("i", keep="first")

# print(f'df.shape={df.shape}')
# print(f'df={df}')

states_all = list(df["i"].values[:n])
if gate in ['cz', 'cnot']:
    ut.print_data(f'state_all_{n_full}_{n_truc}', states_all, 
                    num_each_row=10, n_make_blank_line=50)

np.shape(df_list)=(2, 500, 3)
df_list=[        i                    path      path_len
0     0-0                     0-0  1.000000e+00
3     0-2                     0-2  1.000000e+00
4     2-0                     2-0  1.000000e+00
37    8-2                 0-2,8-2  1.000000e+00
13    2-2                     2-2  1.000000e+00
..    ...                     ...           ...
471  2-66  0-2,8-2,22-2,59-2,2-66  3.848301e-31
416  2-64   0-2,8-2,8-5,5-35,2-64  3.310563e-31
426  1-70       0-2,8-2,12-2,1-70  2.705814e-31
408  1-66       0-2,8-2,1-42,1-66  3.388947e-32
466  0-92            0-2,8-2,0-92  3.217105e-32

[500 rows x 3 columns],         i                    path      path_len
0     0-0                     0-0  1.000000e+00
3     0-2                     0-2  1.000000e+00
4     2-0                     2-0  1.000000e+00
37    8-2                 2-2,8-2  1.000000e+00
13    2-2                     2-2  1.000000e+00
..    ...                     ...           ...
426  1-70       2-2,8-2,

In [7]:
states_all = ut.trunc_by_graph_estimate(n_truc, core_states, drive_term, eval_tot, wd, A, labels=hspace_full,
                                        path_func=ut.all_path_to_core)
if gate in ['cz', 'cnot']:
    ut.print_data(f'state_all_{n_full}_{n_truc}', states_all, 
                    num_each_row=10, n_make_blank_line=50)

states_all_index = [hspace_full.index(i) for i in states_all]
data = states_all_index
ut.print_data(f'state_all_index_{n_full}_{n_truc}', states_all_index,
                    num_each_row=10, n_make_blank_line=50)


state_all_500_500 = np.array([

'0-0', '5-0', '0-2', '2-0', '2-2', '5-1', '5-2', '0-1', '2-1', '0-5' ,
'2-5', '1-0', '1-2', '9-0', '5-4', '2-4', '0-4', '5-5', '1-1', '0-9' ,
'2-9', '4-0', '9-1', '9-2', '1-5', '4-2', '2-12', '0-12', '2-8', '0-8' ,
'2-16', '0-16', '0-13', '2-13', '5-8', '12-0', '2-21', '4-5', '2-20', '2-18' ,
'0-18', '0-21', '5-26', '2-24', '1-8', '0-26', '18-0', '2-26', '9-4', '0-24' ,

'15-0', '4-9', '5-16', '0-20', '1-12', '5-9', '8-0', '2-45', '0-45', '0-25' ,
'2-39', '1-4', '2-25', '0-39', '8-12', '5-20', '5-12', '2-35', '2-33', '0-33' ,
'5-33', '1-16', '15-4', '0-34', '5-34', '2-34', '13-0', '2-46', '5-24', '0-36' ,
'9-8', '2-36', '2-30', '0-52', '2-52', '15-1', '5-21', '1-9', '2-59', '0-59' ,
'2-65', '0-65', '0-42', '2-42', '0-55', '2-55', '8-1', '12-1', '5-35', '1-25' ,

'0-35', '4-8', '5-18', '12-2', '2-53', '9-5', '5-45', '0-57', '18-1', '8-9' ,
'9-24', '4-1', '1-30', '2-57', '0-30', '5-13', '4-4', '5-25', '2-68', '5-52' ,
'0-68', '1-20', '9-16', '5-30', '9-34